# W2 Day1 (04/14 周一): Attention + Transformer 完整手写实现

## 学习目标
- **理论 (1-1.5h)**: MHA: QKV / 缩放点积 / mask; Transformer: MHSA + FFN + LN + 残差; O(n²d) 复杂度分析
- **实践 (2h)**: 手写完整 MHSA + Transformer Encoder/Decoder block
- **产出物**: 可运行的 Transformer 实现 notebook

---

## 目录
1. [缩放点积注意力 (Scaled Dot-Product Attention)](#1)
2. [多头注意力 (Multi-Head Self-Attention)](#2)
3. [位置编码 (Positional Encoding)](#3)
4. [前馈网络 (Feed-Forward Network)](#4)
5. [Transformer Encoder Block](#5)
6. [Transformer Decoder Block (含 Causal Mask + Cross-Attention)](#6)
7. [完整 Transformer 模型](#7)
8. [验证实验: 对比 PyTorch 官方实现](#8)
9. [复杂度分析 & 可视化](#9)
10. [小型翻译任务端到端训练](#10)
11. [思考题 & 总结](#11)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np
import time
import copy

# 确保可复现
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

---
## 1. 缩放点积注意力 (Scaled Dot-Product Attention) <a id='1'></a>

### 1.1 直觉理解

Attention 的核心思想: **让序列中的每个位置都能"看到"其他所有位置，并根据相关性加权聚合信息。**

三个关键角色:
- **Query (Q)**: "我在找什么" —— 当前位置想要的信息
- **Key (K)**: "我有什么" —— 每个位置提供的索引信息
- **Value (V)**: "我的内容" —— 实际要聚合的信息

公式:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

### 1.2 为什么要除以 $\sqrt{d_k}$ ?

当 $d_k$ 很大时，$QK^T$ 的方差会随 $d_k$ 增大。假设 Q, K 的元素都是独立的均值0方差1的随机变量:
$$\text{Var}(q \cdot k) = \sum_{i=1}^{d_k} \text{Var}(q_i \cdot k_i) = d_k$$

除以 $\sqrt{d_k}$ 后方差变回1，防止 softmax 进入梯度极小的饱和区。

In [ ]:
# 实验: 验证为什么需要缩放
print("=" * 60)
print("实验: QK^T 方差随 d_k 增长")
print("=" * 60)

for d_k in [8, 64, 256, 512, 1024]:
    q = torch.randn(1000, d_k)  # 1000个样本
    k = torch.randn(1000, d_k)
    
    # 不缩放
    raw_score = (q * k).sum(dim=-1)  # dot product
    # 缩放后
    scaled_score = raw_score / math.sqrt(d_k)
    
    print(f"d_k={d_k:4d} | 未缩放 Var={raw_score.var():.2f} (理论={d_k}) | "
          f"缩放后 Var={scaled_score.var():.2f} (理论=1.0)")

In [ ]:
# 实验: softmax 在大方差 vs 小方差输入下的行为
print("\n" + "=" * 60)
print("实验: Softmax 饱和效应")
print("=" * 60)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, scale in enumerate([1.0, 10.0, 50.0]):
    logits = torch.randn(1, 10) * scale
    probs = F.softmax(logits, dim=-1)
    axes[i].bar(range(10), probs[0].numpy())
    axes[i].set_title(f'输入方差≈{scale**2:.0f}\nmax prob={probs.max():.4f}')
    axes[i].set_ylim(0, 1.1)
    axes[i].set_xlabel('Position')
    axes[i].set_ylabel('Attention Weight')

plt.suptitle('Softmax 在不同方差输入下的分布 (方差越大→越尖锐→梯度越小)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None, dropout=None):
    """
    手写缩放点积注意力
    
    Args:
        Q: (batch, n_heads, seq_len_q, d_k)
        K: (batch, n_heads, seq_len_k, d_k)
        V: (batch, n_heads, seq_len_k, d_v)
        mask: (batch, 1, 1, seq_len_k) 或 (batch, 1, seq_len_q, seq_len_k)
        dropout: nn.Dropout 或 None
    
    Returns:
        output: (batch, n_heads, seq_len_q, d_v)
        attention_weights: (batch, n_heads, seq_len_q, seq_len_k)
    """
    d_k = Q.size(-1)
    
    # Step 1: QK^T / sqrt(d_k)
    # Q: (..., seq_q, d_k) @ K^T: (..., d_k, seq_k) -> (..., seq_q, seq_k)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    
    # Step 2: 应用 mask (将被 mask 的位置设为 -inf，softmax 后变为 0)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Step 3: Softmax 归一化
    attention_weights = F.softmax(scores, dim=-1)
    
    # Step 4: Dropout (训练时)
    if dropout is not None:
        attention_weights = dropout(attention_weights)
    
    # Step 5: 加权求和 V
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights


# 测试
batch, n_heads, seq_len, d_k = 2, 4, 8, 64
Q = torch.randn(batch, n_heads, seq_len, d_k)
K = torch.randn(batch, n_heads, seq_len, d_k)
V = torch.randn(batch, n_heads, seq_len, d_k)

out, attn_w = scaled_dot_product_attention(Q, K, V)
print(f"输入 Q shape: {Q.shape}")
print(f"输出 shape: {out.shape}")
print(f"注意力权重 shape: {attn_w.shape}")
print(f"注意力权重每行之和 (应为1): {attn_w[0, 0].sum(dim=-1)}")

### 1.3 Mask 的两种用途

1. **Padding Mask**: 忽略填充位置 (batch中长度不一的序列)
2. **Causal Mask (Look-ahead Mask)**: 防止 Decoder 看到未来位置 (自回归生成)

In [ ]:
def create_padding_mask(seq_lens, max_len):
    """
    创建 padding mask
    seq_lens: (batch,) 每个序列的实际长度
    返回: (batch, 1, 1, max_len) 有效位置为1，padding为0
    """
    batch_size = len(seq_lens)
    mask = torch.zeros(batch_size, max_len)
    for i, length in enumerate(seq_lens):
        mask[i, :length] = 1
    return mask.unsqueeze(1).unsqueeze(2)  # (batch, 1, 1, max_len)


def create_causal_mask(seq_len):
    """
    创建因果 mask (下三角矩阵)
    返回: (1, 1, seq_len, seq_len)
    """
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask.unsqueeze(0).unsqueeze(0)  # (1, 1, seq_len, seq_len)


# 可视化 masks
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Padding mask
pad_mask = create_padding_mask([5, 3, 7], max_len=8)
axes[0].imshow(pad_mask[1, 0].numpy(), cmap='Blues', aspect='auto')
axes[0].set_title('Padding Mask (seq_len=3, max=8)\n蓝色=可见, 白色=masked')
axes[0].set_xlabel('Key position')

# Causal mask
causal_mask = create_causal_mask(8)
axes[1].imshow(causal_mask[0, 0].numpy(), cmap='Blues', aspect='auto')
axes[1].set_title('Causal Mask (8×8)\n蓝色=可见, 白色=masked')
axes[1].set_xlabel('Key position')
axes[1].set_ylabel('Query position')

plt.tight_layout()
plt.show()

# 验证 causal mask 的效果
print("\n--- Causal Mask Attention 实验 ---")
Q_test = torch.randn(1, 1, 4, 16)
K_test = torch.randn(1, 1, 4, 16)
V_test = torch.randn(1, 1, 4, 16)
causal = create_causal_mask(4)

_, attn_causal = scaled_dot_product_attention(Q_test, K_test, V_test, mask=causal)
print("Causal attention weights (注意上三角为0):")
print(attn_causal[0, 0].detach().numpy().round(3))

---
## 2. 多头注意力 (Multi-Head Self-Attention, MHSA) <a id='2'></a>

### 2.1 为什么要多头？

单头注意力只能学习一种注意力模式。多头让模型**在不同子空间捕捉不同类型的关系**:
- 某些头可能关注局部语法关系
- 某些头可能关注长距离语义依赖
- 某些头可能关注位置信息

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

关键: 总计算量与单头 d_model 维注意力相同！因为 $d_k = d_{model} / h$

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    手写多头注意力 (从零实现，不使用 nn.MultiheadAttention)
    """
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model 必须能被 n_heads 整除"
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads  # 每个头的维度
        
        # 线性投影层 (可以用一个大矩阵然后 split，这里为了清晰分开写)
        self.W_q = nn.Linear(d_model, d_model, bias=False)  # (d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)  # 输出投影
        
        self.dropout = nn.Dropout(dropout)
        self.attn_weights = None  # 存储注意力权重用于可视化
    
    def split_heads(self, x, batch_size):
        """
        将 (batch, seq_len, d_model) reshape 为 (batch, n_heads, seq_len, d_k)
        """
        x = x.view(batch_size, -1, self.n_heads, self.d_k)
        return x.transpose(1, 2)  # (batch, n_heads, seq_len, d_k)
    
    def forward(self, query, key, value, mask=None):
        """
        Args:
            query: (batch, seq_len_q, d_model)
            key:   (batch, seq_len_k, d_model)
            value: (batch, seq_len_k, d_model)
            mask:  broadcastable to (batch, 1, seq_len_q, seq_len_k)
        
        Returns:
            output: (batch, seq_len_q, d_model)
        """
        batch_size = query.size(0)
        
        # Step 1: 线性投影
        Q = self.W_q(query)  # (batch, seq_q, d_model)
        K = self.W_k(key)    # (batch, seq_k, d_model)
        V = self.W_v(value)  # (batch, seq_k, d_model)
        
        # Step 2: 分头
        Q = self.split_heads(Q, batch_size)  # (batch, n_heads, seq_q, d_k)
        K = self.split_heads(K, batch_size)  # (batch, n_heads, seq_k, d_k)
        V = self.split_heads(V, batch_size)  # (batch, n_heads, seq_k, d_k)
        
        # Step 3: 缩放点积注意力
        attn_output, attn_weights = scaled_dot_product_attention(
            Q, K, V, mask=mask, dropout=self.dropout if self.training else None
        )
        self.attn_weights = attn_weights  # 保存用于可视化
        
        # Step 4: 拼接多头 (batch, n_heads, seq_q, d_k) -> (batch, seq_q, d_model)
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch_size, -1, self.d_model)
        
        # Step 5: 输出投影
        output = self.W_o(attn_output)
        
        return output


# 测试
d_model, n_heads = 512, 8
mha = MultiHeadAttention(d_model, n_heads)

x = torch.randn(2, 10, d_model)  # (batch=2, seq_len=10, d_model=512)
out = mha(x, x, x)  # Self-Attention: Q=K=V=x

print(f"输入 shape: {x.shape}")
print(f"输出 shape: {out.shape}")
print(f"注意力权重 shape: {mha.attn_weights.shape}")
print(f"\n参数量: {sum(p.numel() for p in mha.parameters()):,}")
print(f"理论参数量: 4 × d_model² = 4 × {d_model}² = {4 * d_model**2:,}")

In [ ]:
# 可视化多头注意力权重
mha.eval()
with torch.no_grad():
    # 使用一些有结构的输入来看不同头的模式
    x_vis = torch.randn(1, 12, d_model)
    _ = mha(x_vis, x_vis, x_vis)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(8):
    ax = axes[i // 4, i % 4]
    im = ax.imshow(mha.attn_weights[0, i].detach().numpy(), cmap='hot', aspect='auto')
    ax.set_title(f'Head {i}')
    ax.set_xlabel('Key pos')
    ax.set_ylabel('Query pos')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('8个注意力头的权重分布 (随机初始化)', fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. 位置编码 (Positional Encoding) <a id='3'></a>

### 3.1 为什么需要位置编码？

Attention 是**排列不变的** (permutation invariant):
$$\text{Attention}(\pi(Q), \pi(K), \pi(V)) = \pi(\text{Attention}(Q, K, V))$$

它不知道 token 的顺序！所以必须显式注入位置信息。

### 3.2 Sinusoidal 位置编码

原始论文使用正弦/余弦编码:
$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

优点: 可以外推到更长序列 (理论上)；相对位置有线性变换关系。

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Sinusoidal 位置编码 (与原始 Transformer 论文一致)
    """
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        # 预计算位置编码矩阵
        pe = torch.zeros(max_len, d_model)  # (max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()  # (max_len, 1)
        
        # 使用 log 空间计算 div_term 以提高数值稳定性
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )  # (d_model/2,)
        
        pe[:, 0::2] = torch.sin(position * div_term)  # 偶数维度用 sin
        pe[:, 1::2] = torch.cos(position * div_term)  # 奇数维度用 cos
        
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)  # 不参与梯度更新
    
    def forward(self, x):
        """
        x: (batch, seq_len, d_model)
        """
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# 可视化位置编码
pe_module = PositionalEncoding(d_model=128, dropout=0.0)
pe_values = pe_module.pe[0, :100, :].numpy()  # 前100个位置

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 热力图
im = axes[0].imshow(pe_values.T, cmap='RdBu', aspect='auto')
axes[0].set_xlabel('Position')
axes[0].set_ylabel('Dimension')
axes[0].set_title('位置编码热力图 (d=128, pos=0~99)')
plt.colorbar(im, ax=axes[0])

# 选几个维度画曲线
for dim in [0, 1, 4, 5, 20, 21]:
    axes[1].plot(pe_values[:, dim], label=f'dim={dim}', alpha=0.7)
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Value')
axes[1].set_title('不同维度的位置编码曲线 (低维=高频, 高维=低频)')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 4. 前馈网络 (Position-wise Feed-Forward Network) <a id='4'></a>

$$\text{FFN}(x) = \text{GELU/ReLU}(xW_1 + b_1) W_2 + b_2$$

- 逐位置独立应用 (shared weights across positions)
- 中间层通常扩展到 $4 \times d_{model}$，起到"记忆"的作用
- 原始论文用 ReLU，现代 LLM 常用 GELU 或 SwiGLU

In [ ]:
class FeedForward(nn.Module):
    """
    Position-wise Feed-Forward Network
    """
    def __init__(self, d_model, d_ff=None, dropout=0.1, activation='gelu'):
        super().__init__()
        d_ff = d_ff or 4 * d_model  # 默认 4x 扩展
        
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
        self.activation = {
            'relu': F.relu,
            'gelu': F.gelu,
        }[activation]
    
    def forward(self, x):
        """
        x: (batch, seq_len, d_model)
        """
        return self.linear2(self.dropout(self.activation(self.linear1(x))))


# 测试
ff = FeedForward(d_model=512)
x = torch.randn(2, 10, 512)
out = ff(x)
print(f"FFN 输入: {x.shape} -> 输出: {out.shape}")
print(f"FFN 参数量: {sum(p.numel() for p in ff.parameters()):,}")
print(f"  = d_model × d_ff × 2 + d_ff + d_model = {512*2048*2 + 2048 + 512:,}")

---
## 5. Transformer Encoder Block <a id='5'></a>

### 5.1 关键设计: Pre-LN vs Post-LN

**Post-LN** (原始论文): `x + LayerNorm(Sublayer(x))` → 训练不稳定，需要 warmup

**Pre-LN** (现代标配): `x + Sublayer(LayerNorm(x))` → 训练更稳定，梯度流更好

我们实现 **Pre-LN** 版本 (GPT、LLaMA 等都用这个)。

```
Encoder Block:
    ┌─────────┐
    │  Input x │
    └────┬─────┘
         │
    ┌────▼─────┐
    │ LayerNorm │
    └────┬─────┘
         │
    ┌────▼──────────┐
    │  Multi-Head    │
    │  Self-Attn     │
    └────┬──────────┘
         │
    ┌────▼─────┐
    │ Dropout  │
    └────┬─────┘
         │
    ─────+───── (Residual)
         │
    ┌────▼─────┐
    │ LayerNorm │
    └────┬─────┘
         │
    ┌────▼──────────┐
    │  Feed-Forward  │
    └────┬──────────┘
         │
    ┌────▼─────┐
    │ Dropout  │
    └────┬─────┘
         │
    ─────+───── (Residual)
         │
    ┌────▼─────┐
    │  Output  │
    └──────────┘
```

In [ ]:
class TransformerEncoderBlock(nn.Module):
    """
    Transformer Encoder Block (Pre-LN 版本)
    """
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        
        # Sub-layer 1: Multi-Head Self-Attention
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        
        # Sub-layer 2: Feed-Forward
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)
    
    def forward(self, x, src_mask=None):
        """
        Args:
            x: (batch, seq_len, d_model)
            src_mask: padding mask
        Returns:
            (batch, seq_len, d_model)
        """
        # Pre-LN: LayerNorm -> Sublayer -> Residual
        
        # Sub-layer 1: Self-Attention + Residual
        residual = x
        x = self.norm1(x)
        x = self.self_attn(x, x, x, mask=src_mask)  # Self-Attention: Q=K=V
        x = self.dropout1(x)
        x = residual + x
        
        # Sub-layer 2: FFN + Residual
        residual = x
        x = self.norm2(x)
        x = self.ffn(x)
        x = self.dropout2(x)
        x = residual + x
        
        return x


# 测试
encoder_block = TransformerEncoderBlock(d_model=512, n_heads=8)
x = torch.randn(2, 10, 512)
out = encoder_block(x)
print(f"Encoder Block: {x.shape} -> {out.shape}")
print(f"参数量: {sum(p.numel() for p in encoder_block.parameters()):,}")

In [ ]:
# 实验: Pre-LN vs Post-LN 梯度流对比
print("=" * 60)
print("实验: Pre-LN vs Post-LN 梯度范数随深度变化")
print("=" * 60)

class PostLNEncoderBlock(nn.Module):
    """Post-LN 版本 (原始论文)"""
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)
    
    def forward(self, x, src_mask=None):
        # Post-LN: Sublayer -> Residual -> LayerNorm
        x = self.norm1(x + self.dropout1(self.self_attn(x, x, x, mask=src_mask)))
        x = self.norm2(x + self.dropout2(self.ffn(x)))
        return x

def measure_gradient_norms(block_class, n_layers=12, d_model=256, n_heads=4):
    """测量梯度范数"""
    blocks = nn.ModuleList([block_class(d_model, n_heads, dropout=0.0) for _ in range(n_layers)])
    blocks.train()
    
    x = torch.randn(1, 8, d_model, requires_grad=True)
    
    # 前向
    outputs = [x]
    h = x
    for block in blocks:
        h = block(h)
        outputs.append(h)
    
    # 反向
    loss = h.sum()
    loss.backward()
    
    # 收集每层参数的梯度范数
    grad_norms = []
    for block in blocks:
        norms = [p.grad.norm().item() for p in block.parameters() if p.grad is not None]
        grad_norms.append(np.mean(norms) if norms else 0)
    
    return grad_norms

pre_ln_grads = measure_gradient_norms(TransformerEncoderBlock)
post_ln_grads = measure_gradient_norms(PostLNEncoderBlock)

fig, ax = plt.subplots(figsize=(10, 5))
layers = range(1, len(pre_ln_grads) + 1)
ax.plot(layers, pre_ln_grads, 'b-o', label='Pre-LN (现代)', linewidth=2)
ax.plot(layers, post_ln_grads, 'r-s', label='Post-LN (原始)', linewidth=2)
ax.set_xlabel('Layer')
ax.set_ylabel('Average Gradient Norm')
ax.set_title('Pre-LN vs Post-LN: 梯度范数随深度变化')
ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.show()

print(f"\nPre-LN 梯度范数比 (最深/最浅): {pre_ln_grads[-1]/pre_ln_grads[0]:.4f}")
print(f"Post-LN 梯度范数比 (最深/最浅): {post_ln_grads[-1]/post_ln_grads[0]:.4f}")

---
## 6. Transformer Decoder Block <a id='6'></a>

Decoder 比 Encoder 多一个 **Cross-Attention** 层:

```
Decoder Block:
    Input x (target sequence)
         │
    ┌────▼──────────────┐
    │ Masked Self-Attn  │  ← Causal mask 防止看到未来
    │ + Residual + LN   │
    └────┬──────────────┘
         │
    ┌────▼──────────────┐
    │ Cross-Attention   │  ← Q来自Decoder, K/V来自Encoder
    │ + Residual + LN   │
    └────┬──────────────┘
         │
    ┌────▼──────────────┐
    │ Feed-Forward      │
    │ + Residual + LN   │
    └────┬──────────────┘
         │
       Output
```

In [ ]:
class TransformerDecoderBlock(nn.Module):
    """
    Transformer Decoder Block (Pre-LN 版本)
    包含: Masked Self-Attention + Cross-Attention + FFN
    """
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        
        # Sub-layer 1: Masked Multi-Head Self-Attention
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        
        # Sub-layer 2: Cross-Attention (Encoder-Decoder Attention)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)
        
        # Sub-layer 3: Feed-Forward
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout3 = nn.Dropout(dropout)
    
    def forward(self, x, encoder_output, tgt_mask=None, src_mask=None):
        """
        Args:
            x: (batch, tgt_seq_len, d_model) - decoder input
            encoder_output: (batch, src_seq_len, d_model) - encoder output
            tgt_mask: causal mask for decoder self-attention
            src_mask: padding mask for encoder output
        """
        # Sub-layer 1: Masked Self-Attention
        residual = x
        x = self.norm1(x)
        x = self.self_attn(x, x, x, mask=tgt_mask)  # Q=K=V, 带 causal mask
        x = self.dropout1(x)
        x = residual + x
        
        # Sub-layer 2: Cross-Attention
        residual = x
        x = self.norm2(x)
        # Q 来自 decoder, K/V 来自 encoder
        x = self.cross_attn(x, encoder_output, encoder_output, mask=src_mask)
        x = self.dropout2(x)
        x = residual + x
        
        # Sub-layer 3: FFN
        residual = x
        x = self.norm3(x)
        x = self.ffn(x)
        x = self.dropout3(x)
        x = residual + x
        
        return x


# 测试
decoder_block = TransformerDecoderBlock(d_model=512, n_heads=8)
encoder_out = torch.randn(2, 15, 512)  # encoder 输出 (src_len=15)
tgt = torch.randn(2, 10, 512)          # decoder 输入 (tgt_len=10)
tgt_mask = create_causal_mask(10)       # causal mask

out = decoder_block(tgt, encoder_out, tgt_mask=tgt_mask)
print(f"Decoder Block:")
print(f"  Encoder output: {encoder_out.shape}")
print(f"  Decoder input:  {tgt.shape}")
print(f"  Decoder output: {out.shape}")
print(f"  参数量: {sum(p.numel() for p in decoder_block.parameters()):,}")
print(f"  (比Encoder多一个Cross-Attn: 额外 {4*512*512:,} 参数)")

---
## 7. 完整 Transformer 模型 <a id='7'></a>

将所有组件组装成完整的 Encoder-Decoder Transformer (适用于 seq2seq 任务如机器翻译)。

In [ ]:
class Transformer(nn.Module):
    """
    完整 Encoder-Decoder Transformer
    """
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=512,
        n_heads=8,
        n_encoder_layers=6,
        n_decoder_layers=6,
        d_ff=2048,
        dropout=0.1,
        max_len=5000
    ):
        super().__init__()
        self.d_model = d_model
        
        # Embedding 层
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_len, dropout)
        
        # Encoder
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_encoder_layers)
        ])
        self.encoder_norm = nn.LayerNorm(d_model)  # Pre-LN 需要最后一个 LN
        
        # Decoder
        self.decoder_layers = nn.ModuleList([
            TransformerDecoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_decoder_layers)
        ])
        self.decoder_norm = nn.LayerNorm(d_model)
        
        # 输出层
        self.output_projection = nn.Linear(d_model, tgt_vocab_size)
        
        # 参数初始化
        self._init_parameters()
    
    def _init_parameters(self):
        """Xavier 初始化"""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def encode(self, src, src_mask=None):
        """
        Encoder 前向
        src: (batch, src_len) token indices
        """
        # Embedding + Positional Encoding
        x = self.src_embedding(src) * math.sqrt(self.d_model)  # 论文中的 scaling
        x = self.positional_encoding(x)
        
        # 逐层 Encoder
        for layer in self.encoder_layers:
            x = layer(x, src_mask)
        
        return self.encoder_norm(x)
    
    def decode(self, tgt, encoder_output, tgt_mask=None, src_mask=None):
        """
        Decoder 前向
        tgt: (batch, tgt_len) token indices
        """
        x = self.tgt_embedding(tgt) * math.sqrt(self.d_model)
        x = self.positional_encoding(x)
        
        for layer in self.decoder_layers:
            x = layer(x, encoder_output, tgt_mask, src_mask)
        
        return self.decoder_norm(x)
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        """
        完整前向
        """
        # Encode
        encoder_output = self.encode(src, src_mask)
        
        # Decode
        decoder_output = self.decode(tgt, encoder_output, tgt_mask, src_mask)
        
        # Project to vocab
        logits = self.output_projection(decoder_output)  # (batch, tgt_len, tgt_vocab)
        
        return logits


# 测试完整模型
model = Transformer(
    src_vocab_size=1000,
    tgt_vocab_size=1000,
    d_model=256,
    n_heads=8,
    n_encoder_layers=3,
    n_decoder_layers=3,
    d_ff=1024,
    dropout=0.1
)

# 随机输入
src = torch.randint(0, 1000, (2, 15))   # (batch=2, src_len=15)
tgt = torch.randint(0, 1000, (2, 10))   # (batch=2, tgt_len=10)
tgt_mask = create_causal_mask(10)

logits = model(src, tgt, tgt_mask=tgt_mask)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"完整 Transformer 模型:")
print(f"  src: {src.shape} -> logits: {logits.shape}")
print(f"  总参数量: {total_params:,}")
print(f"  可训练参数量: {trainable_params:,}")
print(f"  约 {total_params/1e6:.1f}M 参数")

In [ ]:
# 参数量分解
print("\n" + "=" * 60)
print("参数量分解")
print("=" * 60)

def count_module_params(model):
    """分模块统计参数量"""
    counts = {}
    for name, module in model.named_children():
        counts[name] = sum(p.numel() for p in module.parameters())
    return counts

param_counts = count_module_params(model)
for name, count in param_counts.items():
    pct = count / total_params * 100
    print(f"  {name:25s}: {count:>10,} ({pct:5.1f}%)")

# 可视化
fig, ax = plt.subplots(figsize=(10, 5))
names = list(param_counts.keys())
values = list(param_counts.values())
colors = plt.cm.Set3(np.linspace(0, 1, len(names)))
bars = ax.barh(names, values, color=colors)
ax.set_xlabel('Parameter Count')
ax.set_title('Transformer 各模块参数量分布')
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + 1000, bar.get_y() + bar.get_height()/2, 
            f'{val:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## 8. 验证实验: 对比 PyTorch 官方实现 <a id='8'></a>

将我们的手写实现与 PyTorch 内置的 `nn.Transformer` 进行对比验证。

In [ ]:
# 对比 PyTorch 官方 MultiheadAttention
print("=" * 60)
print("验证: 手写 MHA vs PyTorch nn.MultiheadAttention")
print("=" * 60)

d_model, n_heads = 256, 8

# 我们的实现
our_mha = MultiHeadAttention(d_model, n_heads, dropout=0.0)

# PyTorch 官方实现
pytorch_mha = nn.MultiheadAttention(d_model, n_heads, dropout=0.0, batch_first=True, bias=False)

# 对齐权重: 将官方的 in_proj_weight 拆分赋给我们的实现
# PyTorch 的 in_proj_weight 是 [W_q; W_k; W_v] 堆叠的
with torch.no_grad():
    wq, wk, wv = pytorch_mha.in_proj_weight.chunk(3, dim=0)
    our_mha.W_q.weight.copy_(wq)
    our_mha.W_k.weight.copy_(wk)
    our_mha.W_v.weight.copy_(wv)
    our_mha.W_o.weight.copy_(pytorch_mha.out_proj.weight)

# 测试输入
x = torch.randn(2, 10, d_model)

our_mha.eval()
pytorch_mha.eval()

with torch.no_grad():
    our_output = our_mha(x, x, x)
    pt_output, _ = pytorch_mha(x, x, x)

diff = (our_output - pt_output).abs()
print(f"最大差异: {diff.max().item():.2e}")
print(f"平均差异: {diff.mean().item():.2e}")
print(f"输出形状一致: {our_output.shape == pt_output.shape}")

if diff.max().item() < 1e-5:
    print("\n✅ 手写实现与 PyTorch 官方实现一致！")
else:
    print("\n⚠️ 存在差异，可能是数值精度问题")

In [ ]:
# 前向传播速度对比
print("\n" + "=" * 60)
print("前向传播速度对比")
print("=" * 60)

x = torch.randn(8, 64, d_model)  # 较大的输入

# Warmup
for _ in range(10):
    _ = our_mha(x, x, x)
    _ = pytorch_mha(x, x, x)

# 我们的实现
n_runs = 100
start = time.time()
with torch.no_grad():
    for _ in range(n_runs):
        _ = our_mha(x, x, x)
our_time = (time.time() - start) / n_runs * 1000

# PyTorch 官方
start = time.time()
with torch.no_grad():
    for _ in range(n_runs):
        _ = pytorch_mha(x, x, x)
pt_time = (time.time() - start) / n_runs * 1000

print(f"我们的实现: {our_time:.2f} ms")
print(f"PyTorch 官方: {pt_time:.2f} ms")
print(f"比率: {our_time/pt_time:.2f}x (>1 说明官方更快)")

---
## 9. 复杂度分析 & 可视化 <a id='9'></a>

### 9.1 时间复杂度

| 操作 | 复杂度 | 说明 |
|------|--------|------|
| QK^T | O(n²d) | n=seq_len, d=d_model/n_heads |
| Softmax | O(n²) | 对每行归一化 |
| Attn × V | O(n²d) | |
| FFN | O(nd_ff) | d_ff = 4d |
| **总计 (一层)** | **O(n²d + nd_ff)** | n²d 是瓶颈！ |

当 n 很大时 (如 n > 4096)，O(n²d) 的自注意力成为主要瓶颈。这就是为什么长序列处理需要线性注意力/稀疏注意力等改进。

In [ ]:
# 实测: Self-Attention 计算时间随序列长度的 O(n²) 增长
print("=" * 60)
print("实测: Attention 时间 vs 序列长度")
print("=" * 60)

d_model_test = 256
n_heads_test = 8
mha_test = MultiHeadAttention(d_model_test, n_heads_test, dropout=0.0)
mha_test.eval()

seq_lengths = [16, 32, 64, 128, 256, 512, 1024]
times = []
memory_usage = []

for seq_len in seq_lengths:
    x = torch.randn(1, seq_len, d_model_test)
    
    # Warmup
    with torch.no_grad():
        for _ in range(5):
            _ = mha_test(x, x, x)
    
    # 计时
    n_runs = 50
    start = time.time()
    with torch.no_grad():
        for _ in range(n_runs):
            _ = mha_test(x, x, x)
    elapsed = (time.time() - start) / n_runs * 1000
    times.append(elapsed)
    
    # 注意力矩阵内存 (n² × n_heads × float32)
    attn_mem = seq_len * seq_len * n_heads_test * 4 / (1024**2)  # MB
    memory_usage.append(attn_mem)
    
    print(f"  seq_len={seq_len:5d} | time={elapsed:8.3f} ms | "
          f"attn_matrix={attn_mem:8.2f} MB")

# 画图
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 时间 vs 序列长度
axes[0].plot(seq_lengths, times, 'bo-', label='实测时间', linewidth=2)
# 拟合 O(n²) 曲线
n_arr = np.array(seq_lengths, dtype=float)
scale = times[0] / (seq_lengths[0]**2)
axes[0].plot(seq_lengths, scale * n_arr**2, 'r--', label='O(n²) 拟合', alpha=0.7)
axes[0].set_xlabel('Sequence Length')
axes[0].set_ylabel('Time (ms)')
axes[0].set_title('Self-Attention 时间 vs 序列长度')
axes[0].legend()
axes[0].set_xscale('log', base=2)
axes[0].set_yscale('log', base=2)

# 内存 vs 序列长度
axes[1].plot(seq_lengths, memory_usage, 'go-', label='Attention 矩阵内存', linewidth=2)
axes[1].set_xlabel('Sequence Length')
axes[1].set_ylabel('Memory (MB)')
axes[1].set_title('Attention 矩阵显存占用 vs 序列长度')
axes[1].legend()
axes[1].set_xscale('log', base=2)
axes[1].set_yscale('log', base=2)

plt.tight_layout()
plt.show()

In [ ]:
# 计算常见模型的注意力矩阵显存
print("\n" + "=" * 60)
print("常见模型 Attention 矩阵显存估算")
print("=" * 60)

configs = [
    ("GPT-2", 1024, 12, 12),
    ("GPT-3 Small", 2048, 12, 12),
    ("LLaMA-7B", 2048, 32, 32),
    ("LLaMA-7B 4K", 4096, 32, 32),
    ("GPT-4 (est.)", 8192, 32, 96),
    ("Claude 100K", 100000, 32, 80),
]

print(f"{'模型':20s} | {'Seq Len':>8s} | {'Layers':>6s} | {'Heads':>5s} | {'显存(单层)':>12s} | {'显存(全部)':>12s}")
print("-" * 80)

for name, seq_len, n_layers, n_heads in configs:
    # 每层: n_heads × seq_len × seq_len × 4 bytes (float32)
    mem_per_layer = n_heads * seq_len * seq_len * 4 / (1024**3)  # GB
    mem_total = mem_per_layer * n_layers
    print(f"{name:20s} | {seq_len:>8,} | {n_layers:>6d} | {n_heads:>5d} | {mem_per_layer:>9.3f} GB | {mem_total:>9.2f} GB")

print("\n💡 这就是为什么 Flash Attention、MQA/GQA、KV Cache 如此重要！")

---
## 10. 小型翻译任务: 端到端训练 <a id='10'></a>

用一个简单的**数字序列反转**任务来验证 Transformer 的学习能力:

输入: `[1, 2, 3, 4, 5]` → 输出: `[5, 4, 3, 2, 1]`

这个任务虽然简单，但能验证:
- Encoder 能否编码整个输入序列
- Decoder 能否用 Cross-Attention 从 Encoder 获取信息
- Causal mask 是否正确工作
- 自回归生成是否正确

In [ ]:
# 数据生成
PAD_TOKEN = 0
SOS_TOKEN = 1  # Start of Sequence
EOS_TOKEN = 2  # End of Sequence
VOCAB_SIZE = 13  # 0=PAD, 1=SOS, 2=EOS, 3-12=数字0-9

def generate_reverse_data(n_samples, seq_len=5):
    """
    生成反转任务数据
    src: [3, 5, 7, 9, 4]      (数字用 3~12 表示)
    tgt: [SOS, 4, 9, 7, 5, 3, EOS]  (加上 SOS/EOS)
    """
    src_data = []
    tgt_data = []
    
    for _ in range(n_samples):
        # 随机长度 (3 到 seq_len)
        length = torch.randint(3, seq_len + 1, (1,)).item()
        seq = torch.randint(3, VOCAB_SIZE, (length,))  # 3~12
        
        # Source: 序列 + padding
        src = torch.zeros(seq_len, dtype=torch.long)
        src[:length] = seq
        
        # Target: SOS + 反转序列 + EOS + padding
        reversed_seq = seq.flip(0)
        tgt = torch.zeros(seq_len + 2, dtype=torch.long)  # +2 for SOS, EOS
        tgt[0] = SOS_TOKEN
        tgt[1:length+1] = reversed_seq
        tgt[length+1] = EOS_TOKEN
        
        src_data.append(src)
        tgt_data.append(tgt)
    
    return torch.stack(src_data), torch.stack(tgt_data)


# 生成数据
train_src, train_tgt = generate_reverse_data(3000, seq_len=6)
val_src, val_tgt = generate_reverse_data(500, seq_len=6)

print("数据示例:")
for i in range(3):
    print(f"  src: {train_src[i].tolist()} -> tgt: {train_tgt[i].tolist()}")

In [ ]:
# 训练循环
def train_transformer():
    # 小模型 (任务简单)
    model = Transformer(
        src_vocab_size=VOCAB_SIZE,
        tgt_vocab_size=VOCAB_SIZE,
        d_model=64,
        n_heads=4,
        n_encoder_layers=2,
        n_decoder_layers=2,
        d_ff=256,
        dropout=0.1
    ).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.98), eps=1e-9)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN)
    
    # 训练
    n_epochs = 30
    batch_size = 64
    train_losses = []
    val_accs = []
    
    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0
        n_batches = 0
        
        # Shuffle
        perm = torch.randperm(len(train_src))
        
        for i in range(0, len(train_src), batch_size):
            batch_idx = perm[i:i+batch_size]
            src = train_src[batch_idx].to(device)
            tgt = train_tgt[batch_idx].to(device)
            
            # Decoder 输入: tgt[:-1] (不含最后一个 token)
            tgt_input = tgt[:, :-1]
            # 目标: tgt[1:] (不含 SOS)
            tgt_output = tgt[:, 1:]
            
            # Causal mask
            tgt_mask = create_causal_mask(tgt_input.size(1)).to(device)
            
            # 前向
            logits = model(src, tgt_input, tgt_mask=tgt_mask)
            
            # 计算 loss
            loss = criterion(
                logits.reshape(-1, VOCAB_SIZE),
                tgt_output.reshape(-1)
            )
            
            # 反向
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
            n_batches += 1
        
        avg_loss = epoch_loss / n_batches
        train_losses.append(avg_loss)
        
        # 验证 (序列完全正确率)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            model.eval()
            correct = 0
            total = 0
            
            with torch.no_grad():
                for i in range(0, len(val_src), batch_size):
                    src = val_src[i:i+batch_size].to(device)
                    tgt = val_tgt[i:i+batch_size].to(device)
                    
                    # 贪心解码
                    predictions = greedy_decode(model, src, max_len=tgt.size(1)-1)
                    
                    # 检查完整序列是否正确
                    tgt_output = tgt[:, 1:]  # 去掉 SOS
                    for j in range(len(src)):
                        pred_seq = predictions[j]
                        tgt_seq = tgt_output[j]
                        # 只比较到 EOS
                        eos_pos = (tgt_seq == EOS_TOKEN).nonzero()
                        if len(eos_pos) > 0:
                            end = eos_pos[0].item() + 1
                            if torch.equal(pred_seq[:end], tgt_seq[:end]):
                                correct += 1
                        total += 1
            
            acc = correct / total * 100
            val_accs.append((epoch, acc))
            print(f"Epoch {epoch+1:3d} | Loss: {avg_loss:.4f} | Val Acc: {acc:.1f}%")
    
    return model, train_losses, val_accs


def greedy_decode(model, src, max_len):
    """
    贪心解码 (自回归生成)
    """
    model.eval()
    batch_size = src.size(0)
    
    # Encode
    encoder_output = model.encode(src)
    
    # 初始化 decoder 输入
    tgt_input = torch.full((batch_size, 1), SOS_TOKEN, dtype=torch.long, device=src.device)
    
    outputs = []
    for step in range(max_len):
        tgt_mask = create_causal_mask(tgt_input.size(1)).to(src.device)
        decoder_output = model.decode(tgt_input, encoder_output, tgt_mask=tgt_mask)
        
        # 取最后一个位置的 logits
        next_token_logits = model.output_projection(decoder_output[:, -1, :])
        next_token = next_token_logits.argmax(dim=-1, keepdim=True)
        
        outputs.append(next_token)
        tgt_input = torch.cat([tgt_input, next_token], dim=1)
    
    return torch.cat(outputs, dim=1)  # (batch, max_len)


print("开始训练...")
model, train_losses, val_accs = train_transformer()

In [ ]:
# 训练曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, 'b-', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('训练 Loss')

epochs_eval, accs = zip(*val_accs)
axes[1].plot(epochs_eval, accs, 'g-o', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Sequence Accuracy (%)')
axes[1].set_title('验证集完整序列准确率')
axes[1].set_ylim(0, 105)

plt.tight_layout()
plt.show()

In [ ]:
# 展示一些预测结果
print("\n" + "=" * 60)
print("预测示例")
print("=" * 60)

model.eval()
test_src, test_tgt = generate_reverse_data(10, seq_len=6)
test_src = test_src.to(device)
test_tgt = test_tgt.to(device)

with torch.no_grad():
    predictions = greedy_decode(model, test_src, max_len=test_tgt.size(1)-1)

for i in range(10):
    src_seq = [x for x in test_src[i].cpu().tolist() if x != PAD_TOKEN]
    tgt_seq = [x for x in test_tgt[i, 1:].cpu().tolist() if x != PAD_TOKEN]  # 去掉 SOS
    pred_seq = predictions[i].cpu().tolist()
    # 截断到 EOS
    if EOS_TOKEN in pred_seq:
        pred_seq = pred_seq[:pred_seq.index(EOS_TOKEN) + 1]
    
    match = "✅" if pred_seq == tgt_seq else "❌"
    print(f"{match} src={src_seq} | expected={tgt_seq} | predicted={pred_seq}")

In [ ]:
# 可视化训练后的注意力权重
print("\n可视化: Cross-Attention 权重 (Decoder 如何关注 Encoder 输出)")

model.eval()
# 使用一个简单的例子
test_input = torch.tensor([[3, 4, 5, 6, 7, 0]], device=device)  # [3,4,5,6,7] 无padding
test_tgt_in = torch.tensor([[SOS_TOKEN, 7, 6, 5, 4, 3, EOS_TOKEN]], device=device)

with torch.no_grad():
    enc_out = model.encode(test_input)
    tgt_mask = create_causal_mask(test_tgt_in.size(1)).to(device)
    _ = model.decode(test_tgt_in, enc_out, tgt_mask=tgt_mask)

# 提取最后一层 decoder 的 cross-attention 权重
cross_attn_weights = model.decoder_layers[-1].cross_attn.attn_weights[0]  # (n_heads, tgt_len, src_len)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
tgt_labels = ['SOS', '7', '6', '5', '4', '3', 'EOS']
src_labels = ['3', '4', '5', '6', '7', 'PAD']

for head in range(4):
    im = axes[head].imshow(
        cross_attn_weights[head].cpu().numpy(), 
        cmap='hot', aspect='auto'
    )
    axes[head].set_xticks(range(len(src_labels)))
    axes[head].set_xticklabels(src_labels)
    axes[head].set_yticks(range(len(tgt_labels)))
    axes[head].set_yticklabels(tgt_labels)
    axes[head].set_title(f'Head {head}')
    axes[head].set_xlabel('Encoder (src)')
    if head == 0:
        axes[head].set_ylabel('Decoder (tgt)')

plt.suptitle('Cross-Attention: Decoder 输出每个位置 attend to Encoder 的哪些位置', fontsize=12)
plt.tight_layout()
plt.show()
print("\n💡 理想情况下，Decoder 输出位置应该关注 Encoder 对应的反转位置")
print("   例如: Decoder 的 '7' (位置1) 应该关注 Encoder 的 '7' (位置4)")

---
## 11. 思考题 & 总结 <a id='11'></a>

### 今日核心知识点回顾

1. **缩放点积注意力**: QKV 三元组 + 缩放 (防止 softmax 饱和) + mask (padding / causal)
2. **多头注意力**: 不同子空间捕捉不同关系，总计算量不变
3. **位置编码**: 注意力的排列不变性 → 需要注入位置信息
4. **Encoder Block**: Pre-LN + Self-Attention + FFN + Residual
5. **Decoder Block**: Pre-LN + Masked Self-Attention + Cross-Attention + FFN
6. **O(n²d) 复杂度**: 长序列的主要瓶颈

### 思考题 (面试高频)

1. **为什么 Transformer 比 RNN 更适合并行训练？** (提示: 计算依赖图)
2. **Pre-LN 为什么比 Post-LN 训练更稳定？** (提示: 梯度流)
3. **Multi-Head Attention 中，如果不做 split_heads，直接用 d_model 维的单头，效果会怎样？** (提示: 秩、表达力)
4. **Cross-Attention 中 Q 来自 Decoder、KV 来自 Encoder 的设计直觉是什么？** (提示: 检索)
5. **如何把 O(n²) 的注意力降到 O(n)？** (提示: Linear Attention, Flash Attention 的思路)

### 预备知识: 明天 CLIP + 对比学习

- 今天的 Transformer Encoder 将作为 CLIP 的图像/文本编码器骨干
- Cross-Attention 的思想也会出现在多模态对齐中
- 建议预习: InfoNCE loss 的基本形式

In [ ]:
# 额外练习: 实现 Decoder-Only Transformer (GPT 风格)
# 这为 W2 Day4 的 nanoGPT 做准备

class DecoderOnlyTransformer(nn.Module):
    """
    GPT 风格的 Decoder-Only Transformer
    没有 Encoder，没有 Cross-Attention
    用 Causal Self-Attention 实现自回归
    """
    def __init__(self, vocab_size, d_model=256, n_heads=8, n_layers=4, 
                 d_ff=1024, max_len=512, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_len, d_model)  # 学习式位置编码 (GPT style)
        self.dropout = nn.Dropout(dropout)
        
        # 只有 Encoder Block (但带 causal mask → 等效于 decoder)
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        
        # Weight tying (GPT-2 技巧: embedding 和 output 共享权重)
        self.head.weight = self.token_embedding.weight
    
    def forward(self, x):
        """
        x: (batch, seq_len) token indices
        """
        B, T = x.shape
        
        # Embedding
        tok_emb = self.token_embedding(x)  # (B, T, d_model)
        pos = torch.arange(T, device=x.device).unsqueeze(0)  # (1, T)
        pos_emb = self.position_embedding(pos)  # (1, T, d_model)
        x = self.dropout(tok_emb + pos_emb)
        
        # Causal mask
        causal_mask = create_causal_mask(T).to(x.device)
        
        # Transformer layers
        for layer in self.layers:
            x = layer(x, src_mask=causal_mask)
        
        x = self.final_norm(x)
        logits = self.head(x)  # (B, T, vocab_size)
        
        return logits
    
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        自回归生成
        idx: (B, T) 初始 token 序列
        """
        for _ in range(max_new_tokens):
            # 前向
            logits = self(idx)  # (B, T, vocab)
            logits = logits[:, -1, :] / temperature  # 只取最后一个位置
            
            # Top-k 采样
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)
        
        return idx


# 测试 Decoder-Only
gpt = DecoderOnlyTransformer(vocab_size=100, d_model=128, n_heads=4, n_layers=2)
x = torch.randint(0, 100, (2, 20))
logits = gpt(x)
print(f"\nDecoder-Only Transformer (GPT style):")
print(f"  输入: {x.shape} -> logits: {logits.shape}")
print(f"  参数量: {sum(p.numel() for p in gpt.parameters()):,}")

# 生成示例
prompt = torch.tensor([[1, 2, 3]], dtype=torch.long)
generated = gpt.generate(prompt, max_new_tokens=10, temperature=0.8, top_k=5)
print(f"  生成: {prompt[0].tolist()} -> {generated[0].tolist()}")
print("\n💡 这就是 nanoGPT 的核心架构！Day4 会完整实现并训练。")

In [ ]:
print("\n" + "=" * 60)
print("W2 Day1 完成! 🎉")
print("=" * 60)
print("""
今日成果:
  ✅ 手写 Scaled Dot-Product Attention (含 mask)
  ✅ 手写 Multi-Head Attention (与 PyTorch 官方验证一致)
  ✅ 手写 Sinusoidal Positional Encoding
  ✅ 手写 Transformer Encoder Block (Pre-LN)
  ✅ 手写 Transformer Decoder Block (含 Cross-Attention)
  ✅ 组装完整 Encoder-Decoder Transformer
  ✅ 在序列反转任务上端到端训练验证
  ✅ 复杂度分析 + 显存估算
  ✅ Pre-LN vs Post-LN 梯度实验
  ✅ Decoder-Only (GPT style) 额外实现

明日预习: CLIP 对比学习
  📖 阅读 CLIP 论文摘要和 Figure 1
  📖 理解 InfoNCE loss 的基本形式
""")